In [0]:
# databricks-feature-engineering==0.7.0
# databricks-sdk==0.38.0

In [0]:
from pyspark.sql import SparkSession, DataFrame, types as T, functions as F
import yaml
import os
from databricks.sdk.runtime import dbutils
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from typing import Any, Dict, List
import yaml
import logging

In [0]:
class GeneralUtils:
    def __init__(self, spark: SparkSession):
        """
        input: spark (SparkSession)
        description: Initialize GeneralUtils with a SparkSession.
        output: None
        """
        self.spark = spark

    def get_variable(self, product_name: str) -> Dict[str, str]:
        """
        input: None
        description: Returns a dictionary of configuration paths and metadata.
        output: Dict with configuration and metadata paths
        """
        return {
            "config_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/workflow/configs/config.yml",
            "config_data_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/src/config/config_data.yml",
            "config_setup_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/src/config/config_setup.yml",
            "config_sys_path": f"/Workspace/AI/{product_name}/{product_name}_model/files",
            "metadata_path": "p_ind_core_ai.metadata.md_ai_object",
            "databricks_asset_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/databricks.yml"
            }
        
    def replace_catalog(self, full_table_name: str, new_schema: str, new_catalog: str) -> str:
        """
        input: full_table_name (str), new_schema (str), new_catalog (str)
        description: Replace the catalog and schema in a full table name.
        output: Modified table name string
        """
        parts = full_table_name.split(".")
        if len(parts) < 3:
            raise ValueError("Format not correct -> catalog.schema.table")
        
        parts[0] = new_catalog
        parts[1] = new_schema
        return ".".join(parts)
    
    def get_list_configs(self, obj: Any) -> List[Any]:
        """
        input: obj (iterable)
        description: Convert an iterable object to a list.
        output: List of elements from the object
        """
        try:
            return [x for x in obj]
        except:
            return []

    def table_reader(self, catalog: str, schema: str, table: str) -> DataFrame:
        """
        input: catalog (str), schema (str), table (str)
        description: Read a table from the specified catalog and schema.
        output: Spark DataFrame
        """
        return spark.read.table(catalog + "." + schema + "." + table)
    
    def master_reader(self, data_path: str, snapshot_date: str) -> DataFrame:
        """
        input: data_path (str), snapshot_date (str)
        description: Read a table and filter by snapshot_date and flg_lastSnap = 1.
        output: Spark DataFrame
        """
        return spark.read.table(data_path).where(f"snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
    
    def load_config(self, config_path: str = None) -> Any:
        """
        input: config_path (str, optional)
        description: Load a YAML configuration file.
        output: Parsed YAML as a Python object
        """
        if config_path is None:
            config_path = os.getenv('ARCHIVER_CONFIG_PATH', 'config.yaml')
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
        
    def _get_widget_param(self, name: str, default: Any)-> Any:
        """
        input: name (str), default (any)
        description: Get a Databricks widget parameter or return default if not found.
        output: Value of the widget or default
        """
        try:
            return dbutils.widgets.get(name)
        except Exception:
            return default
    
    def get_pipeline_parameters(self) -> Dict[str, Any]:
        """
        input: None
        description: Retrieve pipeline parameters from Databricks widgets.
        output: Dictionary of pipeline parameters
        """
        params = {
            "snapshot_date": self._get_widget_param("snapshot_date", "2024-02-01"),
            "cod_layerName": self._get_widget_param("cod_layerName", "METRICS"),
            "runId": self._get_widget_param("runId", "123"),
            "dat_startDate": self._get_widget_param("dat_startDate", "2025-01-01 00:00:00"),
            "cod_pipelineID": self._get_widget_param("cod_pipelineID", "1234"),
            "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        }
        return params

    def get_processor_data(self, cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, cod_processErrorCode, des_processErrorDes) -> List[Any]:
        """
        input: Various process metadata fields
        description: Return a list of processor data fields.
        output: List of processor data values
        """
        from datetime import datetime
        return [cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, cod_processErrorCode, des_processErrorDes]

    def get_lg_processmonitor_metadata(self, catalog: str, database: str, table: str, data: List[Any]) -> DataFrame:
        """
        input: catalog (str), database (str), table (str), data (list)
        description: Create a DataFrame for process monitor metadata.
        output: Spark DataFrame with process monitor schema
        """
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType
        from delta.tables import DeltaTable

        lg_processmonitor_schema = StructType([
            StructField("cod_workflowName", StringType(), True),
            StructField("cod_layerName", StringType(), True),
            StructField("dat_processDate", StringType(), True),
            StructField("dat_processStartDate", StringType(), True),
            StructField("dat_processEndDate", StringType(), True),
            StructField("cod_pipelineID", StringType(), True),            
            StructField("cod_runID", StringType(), True),
            StructField("cod_productName", StringType(), True),
            StructField("cod_flowName", StringType(), True),
            StructField("cod_flowNameSub", StringType(), True),
            StructField("dat_snapshotDate", StringType(), True),
            StructField("cod_processStatus", StringType(), True),
            StructField("cod_processErrorCode", StringType(), True),
            StructField("des_processErrorDes", StringType(), True)
        ])

        return spark.createDataFrame([data], lg_processmonitor_schema)
    
    def get_ds_versioning_data(self, id_deploy, cod_productName, cod_tableName, version, met_version, src, primary_key, fields_to_mask, description, certified, dat_systemDate, dat_startDate, dat_endDate, flg_lastSnap) -> List[Any]:
        """
        input: Various dataset versioning fields
        description: Return a list of dataset versioning data, parsing dates as needed.
        output: List of dataset versioning values
        """
        from datetime import datetime

        fmt = "%Y-%m-%d %H:%M:%S"
        return [
            int(id_deploy),
            str(cod_productName),
            str(cod_tableName),
            str(version),
            str(met_version),
            str(src),
            str(primary_key),
            str(fields_to_mask),
            str(description),
            str(certified),
            datetime.strptime(dat_systemDate, fmt),
            datetime.strptime(dat_startDate, fmt),
            datetime.strptime(dat_endDate, fmt),
            int(flg_lastSnap)
        ]

    def get_dataset_versioning_metadata(self, catalog: str, database: str,  table: str, data: List[Any]) -> DataFrame:
        """
        input: catalog (str), database (str), table (str), data (list)
        description: Create a DataFrame for dataset versioning metadata.
        output: Spark DataFrame with dataset versioning schema
        """
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType, DoubleType

        dataset_versioning_schema = StructType([
            StructField("id_deploy", IntegerType(), True),
            StructField("cod_productName", StringType(), True),
            StructField("cod_tableName", StringType(), True),
            StructField("version", StringType(), True),
            StructField("met_version", StringType(), True),
            StructField("src", StringType(), True),            
            StructField("primary_key", StringType(), True),
            StructField("fields_to_mask", StringType(), True),
            StructField("description", StringType(), True),
            StructField("certified", StringType(), True),
            StructField("dat_systemDate", TimestampType(), True),
            StructField("dat_startDate", TimestampType(), True),
            StructField("dat_endDate", TimestampType(), True),
            StructField("flg_lastSnap", IntegerType(), True)
        ])

        return spark.createDataFrame([data], dataset_versioning_schema)

    def populate_log_date(self, catalog: str, database: str, table: str, metrica: str, cod_runID: str, is_started: bool):
        """
        input: catalog (str), database (str), table (str), metrica (str), cod_runID (str), is_started (bool)
        description: Update process start or end date in the log table.
        output: None
        """
        import pyspark.sql.functions as F 
        from delta.tables import DeltaTable
        from datetime import datetime
        if is_started:
            delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

            condition=(
                f"(dat_processStartDate IS NULL OR dat_processStartDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
            )

            delta_table.update(
                condition=condition,
                set={"dat_processStartDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
            )
        else:
            delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

            condition=(
                f"(dat_processEndDate IS NULL OR dat_processEndDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
            )

            delta_table.update(
                condition=condition,
                set={"dat_processEndDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
            )

    def populate_versioning_metadata(self, catalog: str, database: str, table: str, cod_productName: str, cod_tableName: str, src: str):
        """
        input: catalog (str), database (str), table (str), cod_productName (str), cod_tableName (str), src (str)
        description: Update versioning metadata for a dataset, setting end date and flg_lastSnap.
        output: None
        """
        import pyspark.sql.functions as F 
        from delta.tables import DeltaTable
        from datetime import datetime
        delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

        condition=(
                f"cod_productName = '{cod_productName}' AND cod_tableName = '{cod_tableName}' AND src = '{src}' and flg_lastSnap = 1"
        )

        delta_table.update(
                condition=condition,
                set={"dat_endDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
                     "flg_lastSnap": F.lit(0)}
        )

    def write_table(self, catalog: str, database: str, table: str, df: DataFrame, mode: str):
        """
        input: catalog (str), database (str), table (str), df (DataFrame), mode (str)
        description: Write a DataFrame to a table with schema merge and overwrite options.
        output: None
        """
        df.write.mode(mode).option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{database}.{table}")

    def is_dataframe_empty(df: DataFrame)-> bool:
        """
        input: df (DataFrame)
        description: Check if a DataFrame is empty.
        output: True if empty, False otherwise
        """
        return not df.limit(10).collect()
    
    def set_logging(self):
        """
        input: None
        description: Set up and return a logger with a specific format.
        output: Logger object
        """
        logger = logging.getLogger(__name__)
        logger.setLevel(logging.INFO)
        logger.propagate = False

        if not logger.handlers:

            handler = logging.StreamHandler()

            formatter = logging.Formatter(
                fmt='%(asctime)s - %(levelname)s - %(name)s:%(lineno)d - %(message)s',
                datefmt='%Y-%m-%d %H:%M:%S'
            )

            handler.setFormatter(formatter)
            logger.addHandler(handler)
        return logger

    def date_to_start_month(self, date_to_convert: str) -> str:
        """
        input: date_to_convert (str)
        description: Convert a date string to the first day of its month.
        output: String representing the first day of the month
        """
        dt = datetime.strptime(date_to_convert, "%Y-%m-%d")
        first_of_month = dt.replace(day=1)
        return first_of_month.strftime("%Y-%m-%d")
  
    def replace_month_date(self, data_str: str, n_month: int)-> str:
        """
        input: data_str (str), n_month (int)
        description: Subtract n_month months from a date string.
        output: Resulting date string
        """
        data_input = datetime.strptime(data_str, "%Y-%m-%d")

        date_mod = data_input - relativedelta(months=int(n_month))

        date_output = date_mod.strftime("%Y-%m-%d")
        return date_output
  
    def get_previous_monday_date(self)-> date:
        """
        input: None
        description: Get the date of the previous Monday.
        output: Date object for previous Monday
        """
        today = date.today()
        days_to_sub = (today.weekday() - 0) % 7
        return today - timedelta(days=days_to_sub)
    
    def raise_assert_value(self, df: DataFrame, tabella_sorgente: str, snapshot_date: str):
        """
        input: df (DataFrame), tabella_sorgente (str), snapshot_date (str)
        description: Assert that the DataFrame contains records for the given snapshot_date and flg_lastSnap = 1.
        output: Raises AssertionError if condition is not met
        """
        assert df.limit(1).count() > 0, (
            f"La tabella '{tabella_sorgente}' "
            f"non contiene record per snapshot_date='{snapshot_date}' e flg_lastSnap = 1."
        )

In [0]:
# class GeneralUtils:
#     def __init__(self, spark: SparkSession):
#         self.spark = spark

#     def get_variable(self) -> Dict:
#         return {
#             "config_path": "/Workspace/AI/creri/creri_model/files/workflow/configs/config.yml",
#             "config_data_path": "/Workspace/AI/creri/creri_model/files/src/config/config_data.yml",
#             "config_setup_path": "/Workspace/AI/creri/creri_model/files/src/config/config_setup.yml",
#             "config_sys_path": "/Workspace/AI/creri/creri_model/files",
#             "metadata_path": "p_ind_core_ai.metadata.md_ai_object",
#             "databricks_asset_path": "/Workspace/AI/creri/creri_model/files/databricks.yml"
#             }
        
#     def replace_catalog(self, full_table_name: str, new_schema: str, new_catalog: str) -> str:
#         parts = full_table_name.split(".")
#         if len(parts) < 3:
#             raise ValueError("Format not correct -> catalog.schema.table")
        
#         parts[0] = new_catalog
#         parts[1] = new_schema
#         return ".".join(parts)
    
#     def get_list_configs(self, obj):
#         try:
#             return [x for x in obj]
#         except:
#             return []

#     def table_reader(self, catalog: str, schema: str, table: str) -> DataFrame:
#         return spark.read.table(catalog + "." + schema + "." + table)
    
#     def master_reader(self, data_path: str, snapshot_date: str) -> DataFrame:
#         return spark.read.table(data_path).where(f"snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
    
#     def load_config(self, config_path=None):
#         if config_path is None:
#             config_path = os.getenv('ARCHIVER_CONFIG_PATH', 'config.yaml')
#         with open(config_path, 'r') as f:
#             return yaml.safe_load(f)
        
#     def _get_widget_param(self, name, default):
#         try:
#             return dbutils.widgets.get(name)
#         except Exception:
#             return default
    
#     def get_pipeline_parameters(self):
#         params = {
#             "snapshot_date": self._get_widget_param("snapshot_date", "2024-02-01"),
#             "cod_layerName": self._get_widget_param("cod_layerName", "METRICS"),
#             "runId": self._get_widget_param("runId", "123"),
#             "dat_startDate": self._get_widget_param("dat_startDate", "2025-01-01 00:00:00"),
#             "cod_pipelineID": self._get_widget_param("cod_pipelineID", "1234"),
#             "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
#         }
#         return params


#     def get_processor_data(self, cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, cod_processErrorCode, des_processErrorDes) -> list:
#         from datetime import datetime
#         return [cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, cod_processErrorCode, des_processErrorDes] #iid_processID,


#     def get_lg_processmonitor_metadata(self, catalog, database, table, data) -> DataFrame:
#         from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType
#         from delta.tables import DeltaTable

#         lg_processmonitor_schema = StructType([
#             StructField("cod_workflowName", StringType(), True),
#             StructField("cod_layerName", StringType(), True),
#             StructField("dat_processDate", StringType(), True),
#             StructField("dat_processStartDate", StringType(), True),
#             StructField("dat_processEndDate", StringType(), True),
#             StructField("cod_pipelineID", StringType(), True),            
#             StructField("cod_runID", StringType(), True),
#             StructField("cod_productName", StringType(), True),
#             StructField("cod_flowName", StringType(), True),
#             StructField("cod_flowNameSub", StringType(), True),
#             StructField("dat_snapshotDate", StringType(), True),
#             StructField("cod_processStatus", StringType(), True),
#             StructField("cod_processErrorCode", StringType(), True),
#             StructField("des_processErrorDes", StringType(), True)
#         ]) #StructField("iid_processID", IntegerType(), True),

#         return spark.createDataFrame([data], lg_processmonitor_schema)
    
#     def get_ds_versioning_data(self, id_deploy, cod_productName, cod_tableName, version, met_version, src, primary_key, fields_to_mask, description, certified, dat_systemDate, dat_startDate, dat_endDate, flg_lastSnap) -> list:
#         #iid_processID: int,
#         from datetime import datetime
#         # return [id_deploy, cod_productName, cod_tableName, version, met_version, src, primary_key, fields_to_mask, description, certified, dat_systemDate, dat_startDate, dat_endDate, flg_lastSnap]
#         fmt = "%Y-%m-%d %H:%M:%S"
#         return [
#             int(id_deploy),
#             str(cod_productName),
#             str(cod_tableName),
#             str(version),
#             str(met_version),
#             str(src),
#             str(primary_key),
#             str(fields_to_mask),
#             str(description),
#             str(certified),
#             datetime.strptime(dat_systemDate, fmt),
#             datetime.strptime(dat_startDate, fmt),
#             datetime.strptime(dat_endDate, fmt),
#             int(flg_lastSnap)
#         ]


#     def get_dataset_versioning_metadata(self, catalog, database, table, data) -> DataFrame:
#         from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType, DoubleType

#         dataset_versioning_schema = StructType([
#             StructField("id_deploy", IntegerType(), True),
#             StructField("cod_productName", StringType(), True),
#             StructField("cod_tableName", StringType(), True),
#             StructField("version", StringType(), True),
#             StructField("met_version", StringType(), True),
#             StructField("src", StringType(), True),            
#             StructField("primary_key", StringType(), True),
#             StructField("fields_to_mask", StringType(), True),
#             StructField("description", StringType(), True),
#             StructField("certified", StringType(), True),
#             StructField("dat_systemDate", TimestampType(), True),
#             StructField("dat_startDate", TimestampType(), True),
#             StructField("dat_endDate", TimestampType(), True),
#             StructField("flg_lastSnap", IntegerType(), True)
#         ])

#         return spark.createDataFrame([data], dataset_versioning_schema)

    
#     def populate_log_date(self, catalog, database, table, metrica, cod_runID, is_started: bool):
#         import pyspark.sql.functions as F 
#         from delta.tables import DeltaTable
#         from datetime import datetime
#         if is_started:
#             delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#             condition=(
#                 f"(dat_processStartDate IS NULL OR dat_processStartDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
#             )

#             delta_table.update(
#                 condition=condition,
#                 set={"dat_processStartDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
#             )
#         else:
#             delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#             condition=(
#                 f"(dat_processEndDate IS NULL OR dat_processEndDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
#             )

#             delta_table.update(
#                 condition=condition,
#                 set={"dat_processEndDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
#             )

#     # def populate_monitor_log_date(self, catalog, database, table, metrica, cod_runID, is_started: bool):
#     #     import pyspark.sql.functions as F 
#     #     from delta.tables import DeltaTable
#     #     from datetime import datetime
#     #     if is_started:
#     #         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#     #         condition=(
#     #             f"(dat_processStartDate IS NULL OR dat_processStartDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
#     #         )

#     #         delta_table.update(
#     #             condition=condition,
#     #             set={"dat_processStartDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
#     #         )
#     #     else:
#     #         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#     #         condition=(
#     #             f"(dat_processEndDate IS NULL OR dat_processEndDate = 'null') AND cod_flowName = '{metrica}' AND cod_runID = '{cod_runID}'"
#     #         )

#     #         delta_table.update(
#     #             condition=condition,
#     #             set={"dat_processEndDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))}
#     #         )
#     # def populate_monitor_log_date(self, catalog, database, table, metrica, cod_runID, is_started: bool):
#     #     now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     #     full_table_path = f"{catalog}.{database}.{table}"
        
#     #     target_column = "dat_processStartDate" if is_started else "dat_processEndDate"
        
#     #     sql_query = f"""
#     #         UPDATE {full_table_path}
#     #         SET {target_column} = CAST('{now}' AS TIMESTAMP)
#     #         WHERE cod_flowName = '{metrica}' 
#     #           AND cod_runID = '{cod_runID}'
#     #           AND ({target_column} IS NULL OR CAST({target_column} AS STRING) = 'null')
#     #     """
#     #     self.spark.sql(sql_query)

#     # def populate_log_error(self, catalog, database, table, metrica, snapshot_date, result_state, cod_runID, is_error: bool):
#     #     import pyspark.sql.functions as F
#     #     from datetime import datetime
#     #     from delta.tables import DeltaTable
#     #     if is_error:
#     #         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#     #         condition=(
#     #             f"(cod_processErrorCode IS NULL OR cod_processErrorCode = 'null') AND cod_flowName = '{metrica}' and cod_runID = '{cod_runID}'"
#     #         ) #and dat_snapshotDate = '{snapshot_date}'

#     #         delta_table.update(
#     #             condition=condition,
#     #             set={"cod_processErrorCode": F.lit(result_state)}
#     #         )
#     # def populate_log_error(self, catalog, database, table, metrica, snapshot_date, result_state, cod_runID, is_error: bool):
#     #     if not is_error: return
        
#     #     full_table_path = f"{catalog}.{database}.{table}"
#     #     sql_query = f"""
#     #         UPDATE {full_table_path}
#     #         SET cod_processErrorCode = '{result_state}'
#     #         WHERE cod_flowName = '{metrica}' 
#     #           AND cod_runID = '{cod_runID}'
#     #           AND (cod_processErrorCode IS NULL OR CAST(cod_processErrorCode AS STRING) = 'null')
#     #     """
#     #     self.spark.sql(sql_query)

#     # def populate_log_state(self, catalog, database, table, metrica, snapshot_date, cod_runID, is_error: bool):
#     #     import pyspark.sql.functions as F
#     #     from datetime import datetime
#     #     from delta.tables import DeltaTable
#     #     if is_error:
#     #         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#     #         condition=(
#     #             f"(cod_processStatus IS NULL OR cod_processStatus = 'null' OR cod_processStatus = 'Started') AND cod_flowName = '{metrica}' and cod_runID = '{cod_runID}'"
#     #         ) #and dat_snapshotDate = '{snapshot_date}'

#     #         delta_table.update(
#     #             condition=condition,
#     #             set={"cod_processStatus": F.lit("Failed")}
#     #         )
#     #     else:
#     #         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#     #         condition=(
#     #             f"(cod_processStatus IS NULL OR cod_processStatus = 'null' OR cod_processStatus = 'Started') AND cod_flowName = '{metrica}' and cod_runID = '{cod_runID}'"
#     #         ) #and dat_snapshotDate = '{snapshot_date}'

#     #         delta_table.update(
#     #             condition=condition,
#     #             set={"cod_processStatus": F.lit("Succeeded")}
#     #         )
#     # def populate_log_state(self, catalog, database, table, metrica, snapshot_date, cod_runID, is_error: bool):
#     #     now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     #     full_table_path = f"{catalog}.{database}.{table}"
#     #     status = "Failed" if is_error else "Succeeded"
        
#     #     sql_query = f"""
#     #         UPDATE {full_table_path}
#     #         SET 
#     #             cod_processStatus = '{status}',
#     #             dat_processEndDate = CAST('{now}' AS TIMESTAMP)
#     #         WHERE cod_flowName = '{metrica}' 
#     #           AND cod_runID = '{cod_runID}'
#     #           AND (cod_processStatus IS NULL OR CAST(cod_processStatus AS STRING) = 'null' OR cod_processStatus = 'Started')
#     #     """
#     #     self.spark.sql(sql_query)

#     # def close_process_log(self, catalog, database, table, metrica, cod_runID, is_error: bool, error_msg: str = None):
#     #         now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     #         full_table_path = f"{catalog}.{database}.{table}"
            
#     #         status = "Failed" if is_error else "Succeeded"
            
#     #         # Gestione sicura del messaggio di errore (SQL escape per gli apici singoli)
#     #         if is_error and error_msg:
#     #             safe_error_msg = f"'{str(error_msg).replace(chr(39), chr(39)+chr(39))}'" 
#     #         else:
#     #             safe_error_msg = "NULL"

#     #         sql_query = f"""
#     #             UPDATE {full_table_path}
#     #             SET 
#     #                 cod_processStatus = '{status}',
#     #                 dat_processEndDate = CAST('{now}' AS TIMESTAMP),
#     #                 cod_processErrorCode = {safe_error_msg}
#     #             WHERE cod_flowName = '{metrica}' 
#     #             AND cod_runID = '{cod_runID}'
#     #             AND (cod_processStatus IS NULL OR CAST(cod_processStatus AS STRING) = 'null' OR cod_processStatus = 'Started')
#     #         """
#     #         self.spark.sql(sql_query)


#     def populate_versioning_metadata(self, catalog, database, table, cod_productName, cod_tableName, src):
#         import pyspark.sql.functions as F 
#         from delta.tables import DeltaTable
#         from datetime import datetime
#         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#         condition=(
#                 f"cod_productName = '{cod_productName}' AND cod_tableName = '{cod_tableName}' AND src = '{src}' and flg_lastSnap = 1"
#         )

#         delta_table.update(
#                 condition=condition,
#                 set={"dat_endDate": F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
#                      "flg_lastSnap": F.lit(0)}
#         )


#     def write_table(self, catalog, database, table, df, mode):
#         df.write.mode(mode).option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{database}.{table}")

#     def is_dataframe_empty(df):
#         return not df.limit(10).collect()
    
#     # def set_logging(self):
#     #     logger = logging.getLogger(__name__)
#     #     logger.setLevel(logging.INFO)
#     #     logger.propagate = False

#     #     if not logger.handlers:

#     #         handler = logging.StreamHandler()

#     #         formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s',
#     #                                       datefmt='%Y-%m-%d %H:%M:%S')

#     #         handler.setFormatter(formatter)
#     #         logger.addHandler(handler)
#     #     return logger
#     def set_logging(self):
#         logger = logging.getLogger(__name__)
#         logger.setLevel(logging.INFO)
#         logger.propagate = False

#         if not logger.handlers:

#             handler = logging.StreamHandler()

#             formatter = logging.Formatter(
#                 fmt='%(asctime)s - %(levelname)s - %(name)s:%(lineno)d - %(message)s',
#                 datefmt='%Y-%m-%d %H:%M:%S'
#             )

#             handler.setFormatter(formatter)
#             logger.addHandler(handler)
#         return logger

#     def date_to_start_month(self, date_to_convert):
#       dt = datetime.strptime(date_to_convert, "%Y-%m-%d")
#       first_of_month = dt.replace(day=1)
#       return first_of_month.strftime("%Y-%m-%d")
  
#     def replace_month_date(self, data_str, n_month):
#       data_input = datetime.strptime(data_str, "%Y-%m-%d")

#       date_mod = data_input - relativedelta(months=int(n_month))

#       date_output = date_mod.strftime("%Y-%m-%d")
#       return date_output
  
#     def get_previous_monday_date(self):
#         today = date.today()
#         days_to_sub = (today.weekday() - 0) % 7
#         return today - timedelta(days=days_to_sub)
    
#     def raise_assert_value(self, df, tabella_sorgente, snapshot_date):
#         assert df.limit(1).count() > 0, (
#             f"La tabella '{tabella_sorgente}' "
#             f"non contiene record per snapshot_date='{snapshot_date}' e flg_lastSnap = 1."
#         )

In [0]:
class MetricMaterializer:
    def __init__(self, spark: SparkSession):
        """
        input: spark (SparkSession)
        description: Initialize MetricMaterializer with a SparkSession.
        output: None
        """
        self.spark = spark

    def get_max_version(self, catalog: str, database: str, table: str, snapshot_date: str) -> Any:
        """
        input: catalog (str), database (str), table (str), snapshot_date (str)
        description: Get the minimum version from the table where flg_lastSnap = 1.
        output: max_version (any) or None
        """
        try:
            df = (
                spark.read.table(f"{catalog}.{database}.{table}")\
                .where(f"flg_lastSnap = 1")
            )
            row = df.agg({"version": "min"}).first()
            max_version = row[0]
            return max_version
        except Exception:
            return None
    
    def get_max_iid(self, catalog: str, database: str, table: str) -> int:
        """
        input: catalog (str), database (str), table (str)
        description: Get the maximum iid_processID from the specified table.
        output: max_iid (int)
        """
        try:
            existing_df = spark.read.table(f"{catalog}.{database}.{table}")
            max_iid = existing_df.agg({"iid_processID": "max"}).collect()[0][0]
            return max_iid if max_iid is not None else 0
        except Exception as e:
            return 0
        

    def metric_versioning(self, catalog: str, database: str, table: str, snapshot_date: str, version_flag: str) -> str:
        """
        input: catalog (str), database (str), table (str), snapshot_date (str), version_flag (str)
        description: Update flg_lastSnap and dat_endDate for versioned metrics if records exist for the given snapshot_date.
        output: status message (str)
        """
        if version_flag == "versioned":
            try:
                if spark.table(f"{catalog}.{database}.{table}").where(f"snapshot_date = '{snapshot_date}'").limit(1).count() == 0:
                    pass
                else:
                    spark.sql(f"""UPDATE {catalog}.{database}.{table} SET flg_lastSnap = 0, dat_endDate = date_format(current_timestamp(),  'yyyy-MM-dd HH:mm:ss') WHERE snapshot_date = '{snapshot_date}' and flg_lastSnap = 1""")
                return f"Metric versioning {version_flag}"
            except Exception as e:
                raise
        else:
            return f"Metric versioning {version_flag}"

    def is_metrics_updated(self, catalog: str, database: str, table: str, get_version: Any, snapshot_date: str)-> Any:
        """
        input: catalog (str), database (str), table (str), get_version (any), snapshot_date (str)
        description: Check if the current version matches the provided version for the given snapshot_date.
        output: True, False, or None
        """
        current_version = self.get_max_version(catalog, database, table, snapshot_date)
    
        if current_version is None:
            return None
    
        if current_version == get_version:
            return True
    
        return False
    
    def we_need_a_snapshot_overwrite(self, catalog: str, database: str, table: str, snapshot_overwrite: bool, snapshot_date: str) -> bool:
        """
        input: catalog (str), database (str), table (str), snapshot_overwrite (bool), snapshot_date (str)
        description: Delete records for the given snapshot_date if snapshot_overwrite is True.
        output: True if deleted, False otherwise
        """
        if snapshot_overwrite:
            try:
                spark.sql(f"DELETE FROM {catalog}.{database}.{table} WHERE snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
                return True
            except Exception as e:
                print(f"Error deleting snapshot: {e}")
        else:
            return False
    
    def check_snapshot_overwrite(self, catalog: str, database: str, table: str, snapshot_date: str, snapshot_overwrite: str) -> str:
        """
        input: catalog (str), database (str), table (str), snapshot_date (str), snapshot_overwrite (str)
        description: Delete records for the given snapshot_date if snapshot_overwrite is set, and return a status message.
        output: status message (str)
        """
        if snapshot_overwrite:
            try:
                spark.sql(f"delete from {catalog}.{database}.{table} where snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
                return f"Snapshot {snapshot_date} on table {table} deleted correctly!!"
            except Exception as e:
                return str(e).split(' ')[0]
        else:
            return f"No snapshot deletion required!!"

    def add_tech_columns(self, df: DataFrame, dat_startDate: str, runId: str, version: str, flg_lastSnap: int, des_BusinessKey_list: List[str], snapshot_date: str) -> DataFrame:
        """
        input: df (DataFrame), dat_startDate (str), runId (str), version (str), flg_lastSnap (int), des_BusinessKey_list (list), snapshot_date (str)
        description: Add technical columns to the DataFrame for versioning and tracking.
        output: DataFrame with added columns
        """
        from datetime import date, datetime
        import pyspark.sql.functions as F
        return df\
            .withColumn("snapshot_date", F.lit(snapshot_date))\
            .withColumn("dat_startDate", F.lit(dat_startDate))\
            .withColumn("dat_endDate", F.lit("null"))\
            .withColumn("systemDate", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))\
            .withColumn("runId", F.lit(runId)) \
            .withColumn("version", F.lit(version)) \
            .withColumn("flg_lastSnap", F.lit(flg_lastSnap))\
            .withColumn("des_BusinessKey", F.concat_ws("|", *des_BusinessKey_list))

    def remove_tech_columns(self, df: DataFrame)-> DataFrame:
        """
        input: df (DataFrame)
        description: Remove technical columns from the DataFrame.
        output: DataFrame without technical columns
        """
        return df.drop("dat_startDate", "dat_endDate", "systemDate", 
                   "runId", "version", "flg_lastSnap", "iid_recordID", 
                   "iid_processID", "iid", "des_BusinessKey", "snapshot_date")

    def check_snapshot_data(self, table: str, snapshot_date: str) -> bool:    
        """
        input: table (str), snapshot_date (str)
        description: Check if there is any data for the given snapshot_date in the table.
        output: True if data exists, False otherwise
        """
        try:
            if spark.read.table(table)\
                .where(f"snapshot_date = '{snapshot_date}'").limit(1).collect():
                flag = True
            else:
                flag = False
        except Exception as e:
            flag = False
        return flag

    def populate_metrics_end_date(self, catalog: str, database: str, table: str, snapshot_date: str):
        """
        input: catalog (str), database (str), table (str), snapshot_date (str)
        description: Update dat_endDate to a far future date for records with null end date and the given snapshot_date.
        output: None
        """
        import pyspark.sql.functions as F 
        from delta.tables import DeltaTable
        delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

        condition=(
            f"(dat_endDate == 'null' or dat_endDate IS NULL) and snapshot_date = '{snapshot_date}'"
        )
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ts_9999 = "9999" + ts[4:]
        delta_table.update(
            condition=condition,
            set={"dat_endDate": F.lit(ts_9999)}
        )
    
    def rename_conflicting_columns(self, df: DataFrame, existing_cols: List[str], prefix: str) -> DataFrame:
        """
        input: df (DataFrame), existing_cols (list), prefix (str)
        description: Rename columns in df that conflict with existing_cols by adding a prefix.
        output: DataFrame with renamed columns
        """
        cols = df.columns
        existing_cols_set = set(existing_cols)
        new_cols = [
            F.col(c).alias(f"{c}{prefix}") if c in existing_cols_set else F.col(c)
            for c in cols
        ]
        return df.select(*new_cols)
    
    def get_col_list(self, cod_productname: str, cod_ObjectName: str, metadata_path: str):
        """
        input: cod_productname (str), cod_ObjectName (str), metadata_path (str)
        description: Retrieve a list of field names and types for a given product and object from the metadata table.
        output: List of Row objects with cod_FieldName and cod_Type
        """
        return spark.read.table(metadata_path).where(f"cod_productname = '{cod_productname}' and cod_ObjectName = '{cod_ObjectName}' and (cod_FieldName not in ('iid_RecordID', 'iid_processID', 'iid'))").select("cod_FieldName", "cod_Type").collect()

    def parse_data_type(self, type_str: str)-> str:
        """
        input: type_str (str)
        description: Parse a string representation of a data type and return the corresponding Spark type.
        output: Spark data type or string
        """
        if type_str.startswith('DECIMAL'):
            precision, scale = map(int, type_str[type_str.find('(')+1:type_str.find(')')].split(','))
            return T.DecimalType(precision, scale)
        elif type_str == 'STRING':
            return 'string'
        elif type_str == 'INT':
            return 'int'
        elif type_str == 'LONG':
            return 'long'
        elif type_str == 'BIGINT':
            return 'long'
        elif type_str == 'VARCHAR':
            return 'string'
        elif type_str == 'FLOAT':
            return 'float'
        elif type_str == 'DOUBLE':
            return 'double'
        elif type_str == 'BOOLEAN':
            return 'boolean'
        elif type_str == 'DATE':
            return 'date'
        else:
            raise ValueError(f"Unsupported type: {type_str}")

    def get_schema_fields(self, schema_map: Dict[str, str], df: DataFrame) -> tuple[List[str], List[str]]:
        """
        input: schema_map (dict), df (DataFrame)
        description: Get lists of columns that are common between df and schema_map, and columns only in df.
        output: (common_cols, df_only_cols) (tuple of lists)
        """
        cols = df.columns
        common_cols = [c for c in cols if c in schema_map.keys()]
        df_only_cols = [c for c in cols if c not in schema_map.keys()]
        return common_cols, df_only_cols

    def cast_columns(self, common_cols: List[str], schema_map: Dict[str, str], df: DataFrame) -> DataFrame:
        """
        input: common_cols (list), schema_map (dict), df (DataFrame)
        description: Cast columns in df to types specified in schema_map for the common columns.
        output: DataFrame with casted columns
        """
        return df.select(
                *[F.col(c).cast(schema_map[c]).alias(c) for c in common_cols]
            )
        
    # def column_renaming(self, df_inf_codfiscale_piva: DataFrame, logger: Any) -> DataFrame:
    #     """
    #     input: df_inf_codfiscale_piva (DataFrame), logger (Logger)
    #     description: Rename all columns in df_inf_codfiscale_piva except 'COD_FISCALE_PIVA_CLIIN' by appending '_CLIIN'.
    #     output: DataFrame with renamed columns
    #     """
    #     logger.info("Rename all columns in df_inf_codfiscale_piva except 'COD_FISCALE_PIVA_CLIIN' by appending '_CLIIN'.")
    #     cols = df_inf_codfiscale_piva.columns
    #     for col in cols:
    #         if col not in ["COD_FISCALE_PIVA_CLIIN"]:
    #             df_inf_codfiscale_piva = df_inf_codfiscale_piva.withColumnRenamed(col, f"{col}_CLIIN")
    #     return df_inf_codfiscale_piva
       
    def get_metrics_used(self, config: Any, input_params_json: Dict[str, str]) -> List[Dict[str, Any]]:
        """
        input: config (dict), input_params_json (dict)
        description: Get a list of metrics and targets used based on the type and source metrics in input_params_json.
        output: List of metric/target dicts
        """

        try:
            # NEW CODE
            if 'PREP_METRICS' in input_params_json['type']:
                return [x for x in config.get_config_value('metrics') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config.get_config_value('targets') if x['class_name'] in input_params_json['src_metrics']]
            if 'PREP_CUSTOMER_BASE' in input_params_json['type']:
                return [x for x in config.get_config_value('metrics') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config.get_config_value('targets') if x['class_name'] in input_params_json['src_metrics']]
            if 'PREP_MASTER_TRAINING' in input_params_json['type']:
                return [x for x in config.get_config_value('metrics') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config.get_config_value('targets') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['customer_bases'] if x['class_name'] in input_params_json['src_cb']]
            if 'PREP_MASTER_INFERENCE' in input_params_json['type']:
                return [x for x in config.get_config_value('metrics') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config.get_config_value('targets') if x['class_name'] in input_params_json['src_metrics']] + [x for x in config.get_config_value('customer_bases') if x['class_name'] in input_params_json['src_cb']]
            
        except Exception as e:
            # OLD CODE
            if 'PREP_METRICS' in input_params_json['type']:
                return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']]
            if 'PREP_CUSTOMER_BASE' in input_params_json['type']:
                return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']]
            if 'PREP_MASTER_TRAINING' in input_params_json['type']:
                return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['customer_bases'] if x['class_name'] in input_params_json['src_cb']]
            if 'PREP_MASTER_INFERENCE' in input_params_json['type']:
                return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['customer_bases'] if x['class_name'] in input_params_json['src_cb']]
        

    def get_met_version(self, catalog: str, database: str, metrics_used_list: List[Dict[str, Any]], class_name: str, code_ver: str, snapshot_date: str) -> str:
        """
        input: metrics_used_list (list), class_name (str), code_ver (str), snapshot_date (str)
        description: Generate a version hash based on the versions of metrics used and the code version.
        output: Encrypted version string (str)
        """
        met_version_list = []
        for obj in metrics_used_list:
            print(f"{catalog}.{database}.{obj['name']}")
            met_version = spark.read.table(f"{catalog}.{database}.{obj['name']}")\
                .where(f"snapshot_date = '{snapshot_date}'")\
                    .select(F.max("version").alias("version")).collect()[0]['version']
            met_version_list.append(obj['class_name']+"_"+str(met_version))

        met_version_list.append(class_name+"_"+str(code_ver))
        met_version_list = sorted(met_version_list)
        met_version_list = ["|".join(met_version_list)]
        # return hashlib.sha256(met_version_list[0].encode("utf-8")).hexdigest()[:32] ##
        return hashlib.blake2s(met_version_list[0].encode("utf-8"),digest_size=16).hexdigest()
    


    def get_met_version_encripted(self, metrics_used_list: List[Dict[str, Any]], class_name: str, code_ver: str, catalog_name: str, schema_name: str, metadata_table: str) -> List[str]:
        """
        input: metrics_used_list (list), class_name (str), code_ver (str), catalog_name (str), schema_name (str), metadata_table (str)
        description: Generate a sorted, concatenated version string for metrics and code version.
        output: List containing the concatenated version string
        """
        met_version_list = []
        for obj in metrics_used_list:
            print(f"{catalog_name}.{schema_name}.{obj['class_name']}")
            met_tab_name = obj['name']
            met_version = spark.read.table(f"{catalog_name}.{schema_name}.{metadata_table}")\
                .where(f"cod_tableName = '{met_tab_name}' and flg_lastSnap >= 1")\
                    .select(F.max("version").alias("version")).collect()[0]['version']
            met_version_list.append(obj['class_name']+"_"+str(met_version))

        met_version_list.append(class_name+"_"+str(code_ver))
        met_version_list = sorted(met_version_list)
        met_version_list = ["|".join(met_version_list)]
        return met_version_list

In [0]:
# class MetricMaterializer:
#     def __init__(self, spark: SparkSession):
#         self.spark = spark

#     def get_max_version(self, catalog, database, table, snapshot_date):
#         try:
#             df = (
#                 spark.read.table(f"{catalog}.{database}.{table}")\
#                 .where(f"flg_lastSnap = 1")
#             )

#             row = df.agg({"version": "min"}).first()

#             max_version = row[0]
#             return max_version

#         except Exception:
#             return None
    
#     def get_max_iid(self, catalog, database, table) -> int:
#         try:
#             existing_df = spark.read.table(f"{catalog}.{database}.{table}")
#             max_iid = existing_df.agg({"iid_processID": "max"}).collect()[0][0]
#             return max_iid if max_iid is not None else 0
#         except Exception as e:
#             return 0
        

#     def metric_versioning(self, catalog, database, table, snapshot_date, version_flag) -> str:
#         if version_flag == "versioned":
#             try:
#                 if spark.table(f"{catalog}.{database}.{table}").where(f"snapshot_date = '{snapshot_date}'").limit(1).count() == 0:
#                     pass
#                 else:
#                     spark.sql(f"""UPDATE {catalog}.{database}.{table} SET flg_lastSnap = 0, dat_endDate = date_format(current_timestamp(),  'yyyy-MM-dd HH:mm:ss') WHERE snapshot_date = '{snapshot_date}' and flg_lastSnap = 1""")
#                 return f"Metric versioning {version_flag}"
#             except Exception as e:
#                 raise
#         else:
#             return f"Metric versioning {version_flag}"

#     def is_metrics_updated(self, catalog, database, table, get_version, snapshot_date):
#         current_version = self.get_max_version(catalog, database, table, snapshot_date)
    
#         if current_version is None:
#             return None
    
#         if current_version == get_version:
#             return True
    
#         return False
    
#     def we_need_a_snapshot_overwrite(self, catalog: str, database: str, table: str, snapshot_overwrite: bool, snapshot_date: str) -> bool:
#         if snapshot_overwrite:
#             try:
#                 spark.sql(f"DELETE FROM {catalog}.{database}.{table} WHERE snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
#                 return True
#             except Exception as e:
#                 print(f"Error deleting snapshot: {e}")
#         else:
#             return False
    
#     def check_snapshot_overwrite(self, catalog: str, schema: str, table: str, snapshot_date: str, snapshot_overwrite: str) -> str:
#         if snapshot_overwrite:
#             try:
#                 spark.sql(f"delete from {catalog}.{schema}.{table} where snapshot_date = '{snapshot_date}' and flg_lastSnap = 1")
#                 return f"Snapshot {snapshot_date} on table {table} deleted correctly!!"
#             except Exception as e:
#                 return str(e).split(' ')[0]
#         else:
#             return f"No snapshot deletion required!!"

#     def add_tech_columns(self, df, dat_startDate, runId, version, flg_lastSnap, des_BusinessKey_list, snapshot_date):
#         from datetime import date, datetime
#         import pyspark.sql.functions as F
#         return df\
#             .withColumn("snapshot_date", F.lit(snapshot_date))\
#             .withColumn("dat_startDate", F.lit(dat_startDate))\
#             .withColumn("dat_endDate", F.lit("null"))\
#             .withColumn("systemDate", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))\
#             .withColumn("runId", F.lit(runId)) \
#             .withColumn("version", F.lit(version)) \
#             .withColumn("flg_lastSnap", F.lit(flg_lastSnap))\
#             .withColumn("des_BusinessKey", F.concat_ws("|", *des_BusinessKey_list))

#     def remove_tech_columns(self, df):
#         return df.drop("dat_startDate", "dat_endDate", "systemDate", 
#                    "runId", "version", "flg_lastSnap", "iid_recordID", 
#                    "iid_processID", "iid", "des_BusinessKey", "snapshot_date")

#     def check_snapshot_data(self, table: str, snapshot_date: str) -> bool:    
#         try:
#             if spark.read.table(table)\
#                 .where(f"snapshot_date = '{snapshot_date}'").limit(1).collect():
#                 flag = True
#             else:
#                 flag = False
#         except Exception as e:
#             flag = False
#         return flag

#     def populate_metrics_end_date(self, catalog, database, table, snapshot_date):
#         import pyspark.sql.functions as F 
#         from delta.tables import DeltaTable
#         delta_table = DeltaTable.forName(spark, f"{catalog}.{database}.{table}")

#         condition=(
#             f"(dat_endDate == 'null' or dat_endDate IS NULL) and snapshot_date = '{snapshot_date}'"
#         )

#         ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#         ts_9999 = "9999" + ts[4:]

#         delta_table.update(
#             condition=condition,
#             set={"dat_endDate": F.lit(ts_9999)}
#         )
    
#     def rename_conflicting_columns(self, df, existing_cols, prefix):
#         existing_cols_set = set(existing_cols)
#         new_cols = [
#             F.col(c).alias(f"{c}{prefix}") if c in existing_cols_set else F.col(c)
#             for c in df.columns
#         ]

#         return df.select(*new_cols)
    
#     def get_col_list(self, cod_productname, cod_ObjectName, metadata_path):
#         return spark.read.table(metadata_path).where(f"cod_productname = '{cod_productname}' and cod_ObjectName = '{cod_ObjectName}' and (cod_FieldName not in ('iid_RecordID', 'iid_processID', 'iid'))").select("cod_FieldName", "cod_Type").collect()

#     def parse_data_type(self, type_str):
#         if type_str.startswith('DECIMAL'):
#             precision, scale = map(int, type_str[type_str.find('(')+1:type_str.find(')')].split(','))
#             return T.DecimalType(precision, scale)
#         elif type_str == 'STRING':
#             return 'string'
#         elif type_str == 'INT':
#             return 'int'
#         elif type_str == 'LONG':
#             return 'long'
#         elif type_str == 'BIGINT':
#             return 'long'
#         elif type_str == 'VARCHAR':
#             return 'string'
#         elif type_str == 'FLOAT':
#             return 'float'
#         elif type_str == 'DOUBLE':
#             return 'double'
#         elif type_str == 'BOOLEAN':
#             return 'boolean'
#         elif type_str == 'DATE':
#             return 'date'
#         else:
#             raise ValueError(f"Unsupported type: {type_str}")

#     def get_schema_fields(self, schema_map, df):

#         common_cols = [c for c in df.columns if c in schema_map.keys()]
#         df_only_cols = [c for c in df.columns if c not in schema_map.keys()]

#         return common_cols, df_only_cols

#     def cast_columns(self, common_cols, schema_map, df) -> DataFrame:
#         return df.select(
#                 *[F.col(c).cast(schema_map[c]).alias(c) for c in common_cols]
#             )
        
#     def column_renaming(self, df_inf_codfiscale_piva, logger) -> DataFrame:
#            logger.info("Rename all columns in df_inf_codfiscale_piva except 'COD_FISCALE_PIVA_CLIIN' by appending '_CLIIN'.")
#            for col in df_inf_codfiscale_piva.columns:
#                   if col not in ["COD_FISCALE_PIVA_CLIIN"]:
#                          df_inf_codfiscale_piva = df_inf_codfiscale_piva.withColumnRenamed(col, f"{col}_CLIIN")
#            return df_inf_codfiscale_piva
       
#     def get_metrics_used(self, config, input_params_json) -> list:
#         if 'PREP_METRICS' in input_params_json['type']:
#             return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']]
#         if 'PREP_CUSTOMER_BASE' in input_params_json['type']:
#             return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']]
#         if 'PREP_MASTER_TRAINING' in input_params_json['type']:
#             return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['customer_bases'] if x['class_name'] in input_params_json['src_cb']]
#         if 'PREP_MASTER_INFERENCE' in input_params_json['type']:
#             return [x for x in config['metrics'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['targets'] if x['class_name'] in input_params_json['src_metrics']] + [x for x in config['customer_bases'] if x['class_name'] in input_params_json['src_cb']]

#     def get_met_version(self, metrics_used_list, class_name, code_ver, snapshot_date) -> list:
#         met_version_list = []
#         for obj in metrics_used_list:
#             print(f"{catalog_name}.{schema_name}.{obj['name']}")
#             met_version = spark.read.table(f"{catalog_name}.{schema_name}.{obj['name']}")\
#                 .where(f"snapshot_date = '{snapshot_date}'")\
#                     .select(F.max("version").alias("version")).collect()[0]['version']
#             met_version_list.append(obj['class_name']+"_"+str(met_version))

#         met_version_list.append(class_name+"_"+str(code_ver))
#         met_version_list = sorted(met_version_list)
#         met_version_list = ["|".join(met_version_list)]
#         # return hashlib.sha256(met_version_list[0].encode("utf-8")).hexdigest()[:32] ##
#         return hashlib.blake2s(met_version_list[0].encode("utf-8"),digest_size=16).hexdigest()
    


#     def get_met_version_encripted(self, metrics_used_list, class_name, code_ver, catalog_name, schema_name, metadata_table) -> list:
#         met_version_list = []
#         for obj in metrics_used_list:
#             print(f"{catalog_name}.{schema_name}.{obj['class_name']}")
#             met_tab_name = obj['name']
#             met_version = spark.read.table(f"{catalog_name}.{schema_name}.{metadata_table}")\
#                 .where(f"cod_tableName = '{met_tab_name}' and flg_lastSnap >= 1")\
#                     .select(F.max("version").alias("version")).collect()[0]['version']
#             met_version_list.append(obj['class_name']+"_"+str(met_version))

#         met_version_list.append(class_name+"_"+str(code_ver))
#         met_version_list = sorted(met_version_list)
#         met_version_list = ["|".join(met_version_list)]
#         return met_version_list